# AeroTwin Digital Twin - Colab & InfluxDB Framework

This notebook sets up InfluxDB locally on the fast ephemeral Colab storage, backs it up to your Google Drive, and runs a FastAPI server exposed via ngrok.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/AeroTwin_Influx_Backups"
!pip install -q fastapi uvicorn influxdb-client nest-asyncio pyngrok

In [ ]:
%%bash
wget -q https://dl.influxdata.com/influxdb/releases/influxdb2-2.7.4-linux-amd64.tar.gz
tar -xzf influxdb2-2.7.4-linux-amd64.tar.gz


In [ ]:
%%bash
nohup ./influxdb2-2.7.4-linux-amd64/influxd \
  --bolt-path /content/influx_data/influxd.bolt \
  --engine-path /content/influx_data/engine \
  > influx_db.log 2>&1 &

echo "InfluxDB is starting in the background..."
sleep 5

In [ ]:
%%bash
./influxdb2-2.7.4-linux-amd64/influx setup \
  --username admin \
  --password adminpassword123 \
  --org aerotwin \
  --bucket engine_telemetry \
  --token my-super-secret-auth-token \
  --force

In [ ]:
import os
import time
import shutil
import threading

def backup_to_gdrive():
    source_dir = "/content/influx_data"
    backup_dir = "/content/drive/MyDrive/AeroTwin_Influx_Backups/latest_backup"
    
    while True:
        time.sleep(300) # Backup every 5 minutes
        print("\n[SYSTEM] Starting backup to Google Drive...")
        try:
            if os.path.exists(backup_dir):
                shutil.rmtree(backup_dir)
            shutil.copytree(source_dir, backup_dir)
            print("[SYSTEM] Backup successful!")
        except Exception as e:
            print(f"[SYSTEM] Backup failed: {e}")

backup_thread = threading.Thread(target=backup_to_gdrive, daemon=True)
backup_thread.start()
print("Background backup thread to Google Drive started.")

In [ ]:
import os
import nest_asyncio
import uvicorn
from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel
from pyngrok import ngrok
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

app = FastAPI(title="AeroTwin Colab API")

client = InfluxDBClient(
    url="http://localhost:8086", 
    token="my-super-secret-auth-token", 
    org="aerotwin"
)
write_api = client.write_api(write_options=SYNCHRONOUS)

class EngineTelemetry(BaseModel):
    engine_id: str
    rpm: float
    cht: float 
    egt: float
    vibration: float

def write_to_influx(data: EngineTelemetry):
    point = (
        Point("engine_status")
        .tag("engine_id", data.engine_id)
        .field("rpm", data.rpm)
        .field("cht", data.cht)
        .field("egt", data.egt)
        .field("vibration", data.vibration)
        .time(None, WritePrecision.NS)
    )
    write_api.write(bucket="engine_telemetry", org="aerotwin", record=point)

@app.post("/api/telemetry")
async def ingest_telemetry(data: EngineTelemetry, bg_tasks: BackgroundTasks):
    bg_tasks.add_task(write_to_influx, data)
    return {"status": "success", "message": "Data written to Colab InfluxDB"}

# --- NGROK SETUP ---
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE" # Get from https://dashboard.ngrok.com
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url
print(f"🚀 SEND DATA TO: {public_url}/api/telemetry")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)